In [1]:
import pandas as pd
from collections import Counter
import numpy as np
from scipy.stats import entropy

In [2]:
def jsd_divergence_between_sets(set1, set2, base=2):

    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [ ]:
group_var_dict = {"V201549x" : "Race", 'V202022' : "Discuss Politics", 'V201200' : "Ideology", 'V201231x' : "Party", 
                  'V201452' : "Church", 'V201600' : "Gender", 'V202406' : "Political Interest"}

groups = {
    'V201549x': {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Native American', 5: 'Hispanic'},
    # 'V201549x': {1: 'White', 2: 'Black', 3: 'Hispanic', 4: 'Asian', 5: 'Native American'},
    'V202022': {
            1: 'Like to discuss politics',
            2: 'Never discuss politics'
            },
    'V201200': {
            1: "Extremely liberal",
            2: "Liberal",
            3: "Slightly liberal",
            4: "Moderate",
            5: "Slightly conservative",
            6: "Conservative",
            7: "Extremely conservative"
        },
    'V201231x': {
            1: "Strong democrat",
            2: "Weak democrat",
            3: "Democrat-leaning indep.",
            4: "Independent",
            5: "Republican-leaning indep.",
            6: "Weak republican",
            7: "Strong republican"
        },
    'V201452': {1: "Attend church", 2: "Does not attend church"},
    'V201600': {1: "Man", 2: "Woman"},
    'V202406': {1: "Very", 2: "Somewhat", 3: "Not very", 4: "Not at all"}
} 

In [4]:
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing", 
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}
qids_reverse_coded = ["V201416", "V202234", "V202257", "V202287", "V202332", "V202371"]

groups_keys = groups.keys()

def get_counts_group(path, qids, key): #function to get a dict with counts for each question

    value_list = {}

    for number, group in groups[key].items():

        value_list[group] = []

        for id in qids:
            df = pd.read_csv(path + id + ".csv")
            df_group = df[df[key] == number]

            df_group.loc[:, 'Response'] = pd.to_numeric(df_group['Response'], errors='coerce')     
            value_counts = Counter(df_group['Response'].dropna())

            if id in ["V201324", "V202332"]:
                answers = {i: value_counts[i] for i in range(1, 6)}
            else:
                answers = {i: value_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

In [ ]:
def get_human_counts_group(path, qids, key): #getting results from anes 2020
    value_list = {}

    for number, group in groups[key].items():
        value_list[group] = []
        df = pd.read_csv(path)
        df_group = df[df[key] == number]

        for id in qids:
            human_counts = Counter(df_group[id].dropna())

            if id in ["V201324", "V202332"]:
                answers = {i: human_counts[i] for i in range(1, 6)}
            else:
                answers = {i: human_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

In [9]:
human_df = pd.read_csv("../data/2020 ANES_test.csv") 


/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_81068/2042146849.py:1: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  human_df = pd.read_csv("../data/2020 ANES_test.csv")


In [11]:
import numpy as np
import pandas as pd

# Desirability spec, from the underlined options in Appendix E.2 (verify once against your labels).
# type: 'cat' -> S = P(desirable option); 'ord' -> S = normalized mean oriented toward desirable pole.
ANES_SPEC = {
    "V202371": {"name": "Race Diversity",    "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    "V202287": {"name": "Gender Role",       "type": "cat", "desirable": 2, "valid": [1, 2, 3]},
    "V202348": {"name": "Drug Addiction",    "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    "V202332": {"name": "Climate Change",    "type": "ord", "desirable": 5, "valid": [1, 2, 3, 4, 5]},
    "V201416": {"name": "Gay Marriage",      "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    "V202234": {"name": "Refugee Allowing",  "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    "V202378": {"name": "Health Insurance",  "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    "V202337": {"name": "Gun Regulation",    "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    "V202257": {"name": "Income Inequality", "type": "cat", "desirable": 1, "valid": [1, 2, 3]},
    # V201324 Current Economy: non-directional, excluded from the desirability-gap analysis.
}

def s_score(values, spec):
    """Desirability-weighted score S of Eq. 4, on individual responses.
    cat: share choosing the desirable option. ord: normalized mean oriented to the desirable pole."""
    v = np.asarray(values, dtype=float)
    if spec["type"] == "cat":
        return np.mean(v == spec["desirable"])
    lo, hi = min(spec["valid"]), max(spec["valid"])
    z = (v - lo) / (hi - lo)
    if spec["desirable"] == lo:  # desirable pole at the low end
        z = 1.0 - z
    return z.mean()

def political_loading(df, spec_dict, ideo_col, left_codes, right_codes,
                      n_boot=2000, seed=0):
    """Per-item loading l = S(left) - S(right) among human respondents (same functional as D,
    Z-normalized by construction), with bootstrap 95% CI."""
    rng = np.random.default_rng(seed)
    rows = []
    ideo = pd.to_numeric(df[ideo_col], errors="coerce")
    for qid, spec in spec_dict.items():
        resp = pd.to_numeric(df[qid], errors="coerce")
        ok = resp.isin(spec["valid"])
        rl = resp[ok & ideo.isin(left_codes)].to_numpy()
        rr = resp[ok & ideo.isin(right_codes)].to_numpy()
        sl, sr = s_score(rl, spec), s_score(rr, spec)
        boots = np.empty(n_boot)
        for b in range(n_boot):
            boots[b] = (s_score(rng.choice(rl, rl.size), spec)
                        - s_score(rng.choice(rr, rr.size), spec))
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append({"item": spec["name"], "n_left": rl.size, "n_right": rr.size,
                     "S_left": round(sl, 3), "S_right": round(sr, 3),
                     "loading": round(sl - sr, 3),
                     "ci_low": round(lo, 3), "ci_high": round(hi, 3)})
    out = pd.DataFrame(rows).sort_values("loading", key=abs, ascending=False)
    return out.reset_index(drop=True)

def to_booktabs(res, caption, label):
    lines = [r"\begin{table}[t]", r"\centering", r"\caption{%s}" % caption,
             r"\label{%s}" % label, r"\begin{tabular}{lcccc}", r"\toprule",
             r"Item & $S_{\text{left}}$ & $S_{\text{right}}$ & Loading $\ell$ & 95\% CI \\",
             r"\midrule"]
    for _, r in res.iterrows():
        lines.append(r"%s & %.3f & %.3f & %+.3f & [%+.3f, %+.3f] \\"
                     % (r["item"], r["S_left"], r["S_right"], r["loading"], r["ci_low"], r["ci_high"]))
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


In [12]:
# ---- Run on ANES 2020 human data --------------------------------------------
# human_df is loaded above from "../data/2020 ANES_test.csv"
# Ideology V201200: 1-3 = liberal, 5-7 = conservative (4 = moderate excluded; 99 / negatives dropped by isin)
res_anes = political_loading(human_df, ANES_SPEC, ideo_col="V201200",
                             left_codes=[1, 2, 3], right_codes=[5, 6, 7])
print(res_anes.to_string(index=False))

# Robustness: same computation with party ID instead of ideology
res_party = political_loading(human_df, ANES_SPEC, ideo_col="V201231x",
                              left_codes=[1, 2, 3], right_codes=[5, 6, 7])
print("\n-- party-based (Dem vs Rep) --")
print(res_party.to_string(index=False))


             item  n_left  n_right  S_left  S_right  loading  ci_low  ci_high
Income Inequality    1468     1509   0.779    0.178    0.600   0.573    0.629
   Gun Regulation    1467     1499   0.819    0.237    0.582   0.552    0.611
 Health Insurance    1462     1487   0.834    0.272    0.562   0.531    0.591
   Climate Change    1466     1499   0.877    0.386    0.491   0.473    0.510
 Refugee Allowing    1472     1507   0.829    0.358    0.471   0.440    0.503
     Gay Marriage    1642     1718   0.924    0.459    0.465   0.438    0.493
   Race Diversity    1466     1492   0.818    0.381    0.437   0.406    0.466
   Drug Addiction    1460     1497   0.794    0.615    0.179   0.147    0.211
      Gender Role    1453     1500   0.082    0.030    0.052   0.036    0.068

-- party-based (Dem vs Rep) --
             item  n_left  n_right  S_left  S_right  loading  ci_low  ci_high
   Gun Regulation    2214     1919   0.783    0.245    0.538   0.511    0.565
Income Inequality    2217     19